# Live demo — Swin UNETR spleen (inference only)

מחברת קצרה להרצה **חיה** מול המורה. היא לא מאמנת כלום — טוענת את ה-checkpoint של exp3 ומריצה inference בלבד, אז כל תא רץ תוך שניות עד דקה על ה-GPU של הלאב.

זו **תוספת** ל-`01_data_and_baseline_CLEAN.ipynb` (המחברת המלאה = התיעוד המלא של כל הפרויקט). כאן שלושה דברים שאפשר להריץ על פקודה:

1. **איכות הסגמנטציה** — Dice חי על נפח validation.
2. **ההסבר** — ה-hook על ה-attention + ה-attention-on-spleen ratio, עם תמונה.
3. **ה-null control** — מאומן מול לא-מאומן, שמראה שהריכוז על הטחול *נלמד*.

> רץ **רק על הלאב של שנקר**: צריך את הנתיבים ב-`/home/ori.grossman/...`, את ה-checkpoint של exp3, את דאטה MSD Task09, ו-CUDA. המחברת נשלחת בלי פלט — הפלט נוצר כשמריצים אותה חי. מריצים מלמעלה למטה (Run All), או תא-תא.

In [ ]:
import os, sys, site, glob
us = site.getusersitepackages()
if us not in sys.path: sys.path.insert(0, us)          # after a kernel restart, re-add user-site
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, AsDiscrete, SpatialCrop, ResizeWithPadOrCrop)
from monai.networks.nets import SwinUNETR
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.data import decollate_batch

device    = torch.device("cuda")
PATCH     = (96, 96, 96)
DATA_DIR  = "/home/ori.grossman/nn_final/data/Task09_Spleen"
EXP3_CKPT = "/home/ori.grossman/nn_final/experiments/exp3_cosine_noaug/best.pth"
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# validation preprocessing — identical to the main notebook (RAS, 1.5x1.5x2 mm, HU window, crop)
val_tf = Compose([
    LoadImaged(keys=["image", "label"]), EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-57, a_max=164, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image")])

imgs = sorted(f for f in glob.glob(DATA_DIR + "/imagesTr/*.nii.gz") if not os.path.basename(f).startswith("._"))
lbls = sorted(f for f in glob.glob(DATA_DIR + "/labelsTr/*.nii.gz") if not os.path.basename(f).startswith("._"))
val_files = [{"image": i, "label": l} for i, l in zip(imgs, lbls)][-9:]     # the same 9 validation volumes

model = SwinUNETR(img_size=PATCH, in_channels=1, out_channels=2, feature_size=48, use_checkpoint=True).to(device)
model.load_state_dict(torch.load(EXP3_CKPT, map_location=device)); model.eval()
print("loaded exp3 checkpoint | validation volumes:", len(val_files))

## 1) איכות הסגמנטציה — Dice חי על נפח validation

בוחרים נפח validation אחד, מריצים `sliding_window_inference`, ומודדים Dice מול ה-ground truth. מדגים חי את המספר הראשי (exp3 ≈ **0.9465** בממוצע על 9 הנפחים; לכל נפח בנפרד 0.915–0.966).

In [ ]:
CASE = 0                                    # 0..8 — שנה כדי להראות כל נפח validation
post_pred, post_label = AsDiscrete(argmax=True, to_onehot=2), AsDiscrete(to_onehot=2)
dice_metric = DiceMetric(include_background=False, reduction="mean")

sample = val_tf(val_files[CASE])
x = sample["image"].unsqueeze(0).to(device)
y = sample["label"].unsqueeze(0).to(device)
with torch.no_grad(), torch.autocast("cuda"):
    out = sliding_window_inference(x, PATCH, 4, model, overlap=0.5)
dice_metric(y_pred=[post_pred(o) for o in decollate_batch(out)],
            y=[post_label(o) for o in decollate_batch(y)])
print(f"case {CASE}: Dice = {dice_metric.aggregate().item():.4f}")

## 2) ההסבר — hook על ה-attention + ה-ratio

שמים `forward_hook` על ה-softmax של שכבת ה-`WindowAttention` העמוקה ביותר (מתוך 8), חותכים patch של 96³ סביב הטחול, ומחשבים את ה-**attention-on-spleen ratio** = ממוצע ה-attention בתוך מסכת הטחול חלקי מחוצה לה. מעל 1 = המודל מתרכז באיבר. התמונה: CT / ground truth / prediction / attention.

In [ ]:
def attention_heat(m, patch5d):
    """Hook the deepest WindowAttention softmax, return a 96^3 heat-map + the raw tensor shape."""
    wa = [mod for _, mod in m.named_modules() if type(mod).__name__ == "WindowAttention"]
    store = {}
    h = wa[-1].softmax.register_forward_hook(lambda a, b, o: store.__setitem__("a", o.detach()))
    with torch.no_grad(), torch.autocast("cuda"):
        m(patch5d)
    h.remove()
    a = store["a"][0]; s = round(a.shape[-1] ** (1 / 3))          # 216 tokens -> 6^3
    g = a.float().mean(0).mean(0).reshape(s, s, s)                # mean over heads, then queries
    g = (g - g.min()) / (g.max() - g.min() + 1e-8)
    heat = F.interpolate(g[None, None], size=PATCH, mode="trilinear")[0, 0].cpu().numpy()
    return heat, tuple(store["a"].shape)

# crop a 96^3 patch centred on the spleen (same recipe as the main notebook's attention section)
la = sample["label"][0].cpu().numpy()
center = np.argwhere(la > 0).mean(0).round().astype(int).tolist()
ci = ResizeWithPadOrCrop(PATCH)(SpatialCrop(roi_center=center, roi_size=PATCH)(sample["image"]))
cl = ResizeWithPadOrCrop(PATCH)(SpatialCrop(roi_center=center, roi_size=PATCH)(sample["label"]))
ci5 = ci.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(ci5).argmax(1)[0].cpu().numpy()
heat, ashape = attention_heat(model, ci5)
ct, gt = ci[0].cpu().numpy(), cl[0].cpu().numpy()
ratio = heat[gt > 0].mean() / (heat[gt == 0].mean() + 1e-8)
print("attention tensor:", ashape, "| attention-on-spleen ratio:", round(float(ratio), 2))

z = int(np.argmax(gt.sum((0, 1))))
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(ct[:, :, z], cmap="gray"); ax[0].set_title("CT")
ax[1].imshow(ct[:, :, z], cmap="gray"); ax[1].imshow(np.ma.masked_where(gt[:, :, z] == 0, gt[:, :, z]), cmap="Greens", alpha=.6); ax[1].set_title("ground truth")
ax[2].imshow(ct[:, :, z], cmap="gray"); ax[2].imshow(np.ma.masked_where(pred[:, :, z] == 0, pred[:, :, z]), cmap="Reds", alpha=.6); ax[2].set_title("prediction")
ax[3].imshow(ct[:, :, z], cmap="gray"); ax[3].imshow(heat[:, :, z], cmap="jet", alpha=.5); ax[3].set_title(f"attention (ratio {float(ratio):.2f})")
for a_ in ax: a_.axis("off")
plt.tight_layout(); plt.show()

## 3) null control — האם הריכוז נלמד?

אותה מדידה בדיוק, אבל על מודל **לא-מאומן** (אתחול אקראי, אותה ארכיטקטורה, אותו patch). אם ה-ratio של המאומן גבוה משמעותית מזה של הלא-מאומן, הריכוז על הטחול **נלמד** — הוא לא artifact של הארכיטקטורה או של החיתוך הממורכז. (בהרצה המלאה על כל 9 המקרים: מאומן ≈ **1.90** מול לא-מאומן ≈ **1.11** וקופסה אקראית ≈ 1.28.)

In [ ]:
untrained = SwinUNETR(img_size=PATCH, in_channels=1, out_channels=2, feature_size=48).to(device).eval()
heat_u, _ = attention_heat(untrained, ci5)               # random init, same architecture, same patch
ratio_u = heat_u[gt > 0].mean() / (heat_u[gt == 0].mean() + 1e-8)
print(f"trained ratio   : {float(ratio):.2f}")
print(f"untrained ratio : {float(ratio_u):.2f}")
print("=> concentration is LEARNED" if ratio > ratio_u + 0.3 else "=> inconclusive on this single patch (see the full 9-case control in the main notebook)")